In [1]:
# Instalación de las librerías necesarias

!pip install transformers datasets accelerate --quiet

# Importaciones principales
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import torch
import pandas as pd

# Verificar si hay GPU disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo en uso:", device)


Dispositivo en uso: cuda


Obtener los datos

In [2]:
# Descargamos el archivo directamente desde el GitHub de Amazon Science

!wget https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset/shopping_queries_dataset_products.parquet -O productos_amazon.parquet

# Cargamos los datos con Pandas
df = pd.read_parquet("productos_amazon.parquet")

# Filtramos solo los productos en ESPAÑOL ('es')
# El dataset original tiene inglés y japonés también.
df_es = df[df['product_locale'] == 'es'].copy()

--2025-12-06 14:35:40--  https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset/shopping_queries_dataset_products.parquet
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://media.githubusercontent.com/media/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_products.parquet [following]
--2025-12-06 14:35:41--  https://media.githubusercontent.com/media/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_products.parquet
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1108857465 (1.0G) [application/octet-stream]
Saving to: ‘productos_amazo

Limpiar dataset

In [3]:

# Nos quedamos solo con las columnas originales del parquet
df_es = df[df['product_locale'] == 'es'][['product_title', 'product_description']].dropna().copy()

# Renombramos para facilitar trabajo
df_es.columns = ['Producto', 'Descripcion']

# Reducimos el dataset a 3,000 filas
df_small = df_es.sample(3000, random_state=42)

# Limpiar texto
import re

def limpiar_texto(texto):
    texto = re.sub(r"<.*?>", " ", texto)  # elimina etiquetas html
    texto = texto.replace("", " ")
    texto = texto.replace("\n", " ")
    texto = texto.replace("\r", " ")
    texto = " ".join(texto.split())
    return texto

df_small["Producto"] = df_small["Producto"].apply(limpiar_texto)
df_small["Descripcion"] = df_small["Descripcion"].apply(limpiar_texto)

# Guardar dataset final
df_small.to_csv("dataset_ecommerce_espanol.csv", index=False)

print(f"Dataset reducido y limpio creado con {len(df_small)} filas.")
df_small.head()


Dataset reducido y limpio creado con 3000 filas.


,Producto,Descripcion
143968,Loowa - Objetivo de 24 mm f/14 para Pentax K,Aunque el objetivo de prueba LAOWA es un aspec...
1505255,Juego de Sábanas 3 Piezas para Cama 90x190/200...,Juego de sábanas de 3 piezas compuesto de: 1 s...
143150,Chely Intermarket | 13B2K | Marco de Fotos 18x...,"Marco Chely Intermarket, su pared merece un ma..."
61286,"UGREEN Cable USB C 90 Grados, Cable USB A 2.0 ...",CARGA RÁPIDA Y SINCRONIZACIÓN: Admite salida d...
1184222,Amazon Brand - Umi Funda de Cojin Decoracion p...,Descripción del producto del Umi juego de 2 co...


Tokenización

In [4]:
# Cargamos el tokenizer del modelo GPT-2 en español
model_name = "datificate/gpt2-small-spanish"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 no tiene token de padding por defecto, se lo asignamos
tokenizer.pad_token = tokenizer.eos_token

# Cargamos el CSV procesado
df = pd.read_csv("dataset_ecommerce_espanol.csv")

# Combinamos título y descripción en un solo texto
# Esto ayuda a que el modelo aprenda el estilo completo de escritura
df["texto"] = df["Producto"] + " - " + df["Descripcion"]

# Convertimos el DataFrame a una lista de diccionarios
dataset_dict = {"text": df["texto"].tolist()}

# Creamos un dataset en formato HuggingFace desde la lista
# Este formato permite tokenizarlo, dividirlo y entrenarlo más fácilmente
dataset = load_dataset("text", data_files={"train": "dataset_ecommerce_espanol.csv"})

# Función de tokenización aplicada a cada ejemplo
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=64
    )

# Aplicamos la tokenización a todo el dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("Tokenización completada.")
tokenized_dataset["train"][0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3001 [00:00<?, ? examples/s]

Tokenización completada.


{'input_ids': [5111, 17236, 12, 1630, 10819, 357],
 'attention_mask': [1, 1, 1, 1, 1, 1]}

Cargar el modelo

In [5]:
# Nombre del modelo base en español
model_name = "datificate/gpt2-small-spanish"

# Cargamos el modelo GPT-2 ya preentrenado en español
model = GPT2LMHeadModel.from_pretrained(model_name)

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Entrenamiento

In [6]:
# Ajustamos el token de padding para evitar errores durante el entrenamiento
model.resize_token_embeddings(len(tokenizer))

# El DataCollator se encargará de crear automáticamente los labels
# y manejar el padding para el entrenamiento de lenguaje
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Dividimos el dataset tokenizado en entrenamiento y validación
tokenized_dataset = tokenized_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

print("\nTamaño del dataset de entrenamiento:", len(tokenized_dataset["train"]))
print("Tamaño del dataset de validación:", len(tokenized_dataset["test"]))


Tamaño del dataset de entrenamiento: 2700
Tamaño del dataset de validación: 301


In [7]:
# Definimos los argumentos del entrenamiento
training_args = TrainingArguments(
    output_dir="modelo_final",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

print("Modelo y configuraciones cargadas correctamente.")

Modelo y configuraciones cargadas correctamente.


In [8]:
# Entrenamiento del modelo con Trainer

# Creamos el objeto Trainer con todos los componentes necesarios
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)


# Iniciamos el entrenamiento
trainer.train()

# Guardamos el modelo una vez finalizado el entrenamiento
trainer.save_model("modelo_final")
tokenizer.save_pretrained("modelo_final")

print("Entrenamiento finalizado y modelo guardado.")


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.980500,4.710741
2,4.640000,4.615471
3,4.536700,4.588480


Entrenamiento finalizado y modelo guardado.


Generar texto con el modelo entrenado

In [9]:
from transformers import pipeline

# Cargamos el pipeline usando tu modelo final
generador = pipeline(
    "text-generation",
    model="modelo_final",
    tokenizer="modelo_final",
    device=0 if torch.cuda.is_available() else -1
)

# Ejemplo de entrada: nombre del producto
entrada = "Zapatos deportivos para correr"

resultado = generador(
    entrada,
    max_new_tokens=128,
    num_return_sequences=1,
    temperature=0.7,      # Controla la creatividad
    top_p=0.9, # Aumenta la diversidad de vocabulario
    no_repeat_ngram_size=3 # Evita la repetición de frases largas
)

print("Entrada:", entrada)
print("Descripción generada:")
print(resultado[0]["generated_text"])

Device set to use cuda:0


Entrada: Zapatos deportivos para correr
Descripción generada:
Zapatos deportivos para correr, correr, etc. - Pulseras deportivas para coches de carretera, motocicletas, etc., (10 unidades) - Sabor de coche deportivo (10)","¿Por qué elegirnos las bolsas de coche para competir? ¿Estar una bolsa de coche en el bolsillo de la mochila? ¿Por qué no elegir una bolsa? ¿Cómo usarlas? ¿Qué hacerás? ¿Evitar bolsas para ir a la escuela? ¿Cuál es el dinero? ¿Oye-go? ¿No puedes comprar bolsas de coches? ¿Te gusta hacer bolsas de automóvil? ¿Agrade los bolsas de automóviles? ¿


Gradio

In [10]:
# Instalar Gradio para la interfaz de usuario
!pip install gradio

In [11]:
import gradio as gr
from transformers import pipeline, GPT2Tokenizer
import torch


# Ruta a la carpeta del modelo
MODEL_PATH = "modelo_final"

# Cargamos el tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

# Cargamos el modelo en el pipeline
device_idx = 0 if torch.cuda.is_available() else -1
generador = pipeline(
    "text-generation",
    model=MODEL_PATH,
    tokenizer=tokenizer,
    device=device_idx
)

# Función de Generación de la Interfaz
def generar_descripcion(nombre_producto):
    """Genera la descripción usando los parámetros optimizados."""

    # Formato de entrada que el modelo espera: "Producto: [Nombre]"
    prompt = f"Producto: {nombre_producto}"

    # Parámetros de generación optimizados
    resultado = generador(
        prompt,
        max_new_tokens=128,
        num_return_sequences=1,
        temperature=0.8,         # Creatividad controlada
        top_p=0.9,               # Diversidad de vocabulario
        no_repeat_ngram_size=4   # Evita bucles y repeticiones largas
    )

    # Limpiamos el texto: solo devolvemos el texto generado, no la entrada
    texto_generado = resultado[0]["generated_text"].replace(prompt, "").strip()

    return texto_generado

# Interfaz de Gradio
iface = gr.Interface(
    fn=generar_descripcion,
    inputs=gr.Textbox(lines=1, placeholder="Ej: Zapatos deportivos para correr"),
    outputs=gr.Textbox(label="Descripción de Producto Generada (GPT-2 Fine-tuned)", lines=5),
    title="Generador de Descripciones de E-commerce en Español",
    description="Sistema de IA para crear textos de marketing persuasivos a partir del nombre del producto. Usa el modelo entrenado en el Experimento 2."
)

# Lanzamiento de la aplicación y generación del enlace público
# El argumento `share=True` es el que genera el enlace público temporal.
print("Lanzando la aplicación Gradio...")
iface.launch(share=True)

Device set to use cuda:0


Lanzando la aplicación Gradio...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://830582cf76c6f49df4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
